In [ ]:
#This script downloads buildings from the Microsoft Building Footprints global dataset for the Nigeria area, limiting itself to the tiles that intersect the provided LPG points. 
#After reprojecting the points to UTM, a 150‑meter buffer is applied and only the buildings falling within these buffers are selected. 
#Finally, the code counts how many points have at least one nearby building and results can be saved in separate gpkg files for further analysis.
#takes around 3h for the run

import geopandas as gpd
import pandas as pd
import requests
import gzip
import io
import mercantile
from shapely.geometry import box
import numpy as np


DATA_DIR = "dataset_big"

# 1. Load your LPG points (EPSG:3857) and convert to WGS84
points_file = f"{DATA_DIR}/lpg_points_maps_nigeria_3857.gpkg" #you may change it
gdf_points = gpd.read_file(points_file)
print(f"Loaded {len(gdf_points)} points from {points_file}")

# Reproject points to WGS84 (EPSG:4326) – we'll use this for tile bounding box checks
gdf_points_wgs84 = gdf_points.to_crs(epsg=4326)

# 2. Define Nigeria bounding box (approximate)
west, south, east, north = 2.15, 3.65, 13.78, 13.96

# 3. Get all zoom-9 quadkeys covering Nigeria
quadkeys_nigeria = set()
for tile in mercantile.tiles(west, south, east, north, zooms=9):
    quadkeys_nigeria.add(mercantile.quadkey(tile))
print(f"Nigeria is covered by {len(quadkeys_nigeria)} zoom-9 tiles.")

# 4. Load Microsoft dataset links CSV
csv_url = "https://minedbuildings.z5.web.core.windows.net/global-buildings/dataset-links.csv"
df_links = pd.read_csv(csv_url, dtype=str)
print(f"Total tiles in global dataset: {len(df_links)}")

# Filter to only tiles that intersect Nigeria
tiles_to_download = df_links[df_links['QuadKey'].isin(quadkeys_nigeria)]
print(f"Tiles to process: {len(tiles_to_download)}")

# 5. Helper: get UTM zone EPSG code from longitude
def epsg_utm_from_lon(lon):
    """
    Return EPSG code for the UTM zone containing the given longitude.
    Nigeria is in northern hemisphere, so zones are 326xx.
    """
    zone = int(np.floor((lon + 180) / 6)) + 1
    return f"EPSG:326{zone:02d}"

# 6. Process each tile, collect building footprints near points
building_list = []          # will hold GeoDataFrames of buildings near points
total_buildings = 0
counting= 0
for counting, row in tiles_to_download.iterrows():
    quadkey = row['QuadKey']
    url = row['Url']
    print(f"\nProcessing tile {quadkey} ({counting+1}/{len(tiles_to_download)})")

    # Download the tile
    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code != 200:
            print(f"  -> Download failed (status {resp.status_code}), skipping")
            continue
    except Exception as e:
        print(f"  -> Request error: {e}, skipping")
        continue

    # Read the compressed GeoJSONL
    try:
        with gzip.open(io.BytesIO(resp.content), 'rt', encoding='utf-8') as f:
            gdf_buildings = gpd.read_file(f, driver='GeoJSON')
    except Exception as e:
        print(f"  -> Error reading GeoJSON: {e}, skipping")
        continue

    if gdf_buildings.empty:
        print("  -> No buildings in this tile, skipping")
        continue

    # Buildings are in EPSG:4326
    gdf_buildings = gdf_buildings.set_crs(epsg=4326, allow_override=True)

    # Get tile bounding box to filter points that lie in this tile
    tile_bounds = mercantile.bounds(mercantile.quadkey_to_tile(quadkey))
    tile_bbox = box(tile_bounds.west, tile_bounds.south, tile_bounds.east, tile_bounds.north)

    # Select points inside this tile
    points_in_tile = gdf_points_wgs84[gdf_points_wgs84.intersects(tile_bbox)]
    if points_in_tile.empty:
        print("  -> No LPG points in this tile, skipping")
        continue

    # Determine UTM zone for this tile (based on tile centroid)
    centroid_lon = (tile_bounds.west + tile_bounds.east) / 2
    utm_crs = epsg_utm_from_lon(centroid_lon)
    print(f"  -> Using UTM CRS: {utm_crs}")

    # Project buildings to UTM
    gdf_buildings_utm = gdf_buildings.to_crs(utm_crs)

    # Project the points_in_tile to the same UTM
    points_in_tile_utm = points_in_tile.to_crs(utm_crs)

    # Buffer points by 150 metres (in projected CRS)
    buffered_points = points_in_tile_utm.buffer(150)
    # Make a GeoDataFrame from buffered points (we need geometry for sjoin)
    gdf_buffers = gpd.GeoDataFrame(geometry=buffered_points, crs=utm_crs)

    # Spatial join: find buildings that intersect any buffer
    buildings_near = gpd.sjoin(
        gdf_buildings_utm,
        gdf_buffers[['geometry']],
        how='inner',
        predicate='intersects'
    )

    # Drop the 'index_right' column added by sjoin (it's the buffer index)
    if 'index_right' in buildings_near.columns:
        buildings_near = buildings_near.drop(columns=['index_right'])

    if not buildings_near.empty:
        # Convert back to WGS84 for consistent storage
        buildings_near_wgs84 = buildings_near.to_crs(epsg=4326)
        building_list.append(buildings_near_wgs84)
        count = len(buildings_near_wgs84)
        total_buildings += count
        print(f"  -> Found {count} buildings near points in this tile. Total so far: {total_buildings}")
    else:
        print("  -> No buildings near points in this tile.")

# 7. Combine all collected buildings and save
if building_list:
    gdf_all_buildings = pd.concat(building_list, ignore_index=True)
    # Drop duplicate geometries (unlikely, but just in case)
    gdf_all_buildings = gdf_all_buildings.drop_duplicates(subset='geometry')
    print(f"\nTotal unique buildings near points: {len(gdf_all_buildings)}")

    # Save to GeoPackage
    output_file = f"{DATA_DIR}/buildings_near_lpg_points_150m.gpkg"
    gdf_all_buildings.to_file(output_file, driver='GPKG')
    print(f"Saved buildings to {output_file}")
else:
    print("\nNo buildings found near any LPG point.")

Loaded 4264 points from lpg_points_maps_nigeria_3857.gpkg
Nigeria is covered by 272 zoom-9 tiles.
Total tiles in global dataset: 30344
Tiles to process: 329

Processing tile 122202323 (310/329)
  -> No LPG points in this tile, skipping

Processing tile 122220231 (313/329)
  -> Using UTM CRS: EPSG:32631
  -> No buildings near points in this tile.

Processing tile 122220321 (314/329)
  -> Using UTM CRS: EPSG:32631
  -> No buildings near points in this tile.

Processing tile 122220330 (315/329)
  -> Using UTM CRS: EPSG:32631
  -> No buildings near points in this tile.

Processing tile 122220332 (316/329)
  -> No LPG points in this tile, skipping

Processing tile 122220333 (317/329)
  -> Using UTM CRS: EPSG:32631
  -> No buildings near points in this tile.

Processing tile 122222111 (318/329)
  -> Using UTM CRS: EPSG:32631
  -> No buildings near points in this tile.

Processing tile 122222113 (319/329)


C:\Users\matti\miniconda3\envs\ox\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


  -> Using UTM CRS: EPSG:32631
  -> No buildings near points in this tile.

Processing tile 122223102 (320/329)
  -> Using UTM CRS: EPSG:32632
  -> No buildings near points in this tile.

Processing tile 122223120 (321/329)
  -> Using UTM CRS: EPSG:32632
  -> No buildings near points in this tile.

Processing tile 122223121 (322/329)
  -> Using UTM CRS: EPSG:32632
  -> No buildings near points in this tile.

Processing tile 122202231 (4556/329)
  -> No LPG points in this tile, skipping

Processing tile 122202233 (4558/329)
  -> No LPG points in this tile, skipping

Processing tile 122202320 (4559/329)
  -> No LPG points in this tile, skipping

Processing tile 122202322 (4560/329)
  -> No LPG points in this tile, skipping

Processing tile 122202323 (4561/329)
  -> No LPG points in this tile, skipping

Processing tile 122220011 (4565/329)
  -> No LPG points in this tile, skipping

Processing tile 122220013 (4567/329)
  -> No LPG points in this tile, skipping

Processing tile 122220031 (4

In [ ]:
# 8. Analisi: quanti punti hanno edifici vicini?-

punti_totali = len(gdf_points)
print(f"Punti LPG totali: {punti_totali}")

if building_list:  # se sono stati trovati edifici
    # Estrai gli ID unici dei punti che hanno edifici vicini
    # (dobbiamo ricostruire questa informazione)
    
    # Unisci tutti gli edifici trovati in un unico GeoDataFrame
    gdf_edifici_trovati = pd.concat(building_list, ignore_index=True)
    
    # Spatial join: punti che intersecano gli edifici (buffer 150m già applicato)
    # Usiamo i punti in WGS84 e gli edifici in WGS84
    punti_con_edifici = gpd.sjoin(
        gdf_points_wgs84,  # i punti in WGS84
        gdf_edifici_trovati[['geometry']],  # solo geometria edifici
        how='inner',
        predicate='intersects'
    )
    
    # Numero di punti unici che hanno almeno un edificio
    punti_con_edifici_unici = punti_con_edifici.index.nunique()
    punti_senza_edifici = punti_totali - punti_con_edifici_unici
    
    print(f"\nPunti CON almeno un edificio vicino (≤150m): {punti_con_edifici_unici}")
    print(f"Punti SENZA edifici vicini: {punti_senza_edifici}")
    print(f"Percentuale con edifici: {punti_con_edifici_unici/punti_totali*100:.1f}%")
    
    
else:
    print("\n❌ Nessun edificio trovato vicino a nessun punto LPG")
    print(f"Tutti i {punti_totali} punti sono senza edifici nel raggio di 150m")

Punti LPG totali: 4264

Punti CON almeno un edificio vicino (≤150m): 1294
Punti SENZA edifici vicini: 2970
Percentuale con edifici: 30.3%
